# 06 - Auditoría Manual de Punto de Equilibrio (Break-Even)
**Trabajo de Fin de Máster (TFM)**

Este notebook permite auditar paso a paso los cálculos de recuperación de inversión (CAPEX) para la flota de transporte pesado (44t). A diferencia de otros modelos, aquí los parámetros se definen **manualmente** para facilitar la validación directa de las fórmulas.

### Regla de Precisión
*   Todos los resultados se muestran con **1 solo decimal**.


In [1]:
import pandas as pd
import numpy as np

def format_1_dec(val):
    return f"{val:,.1f}"

# Configuración de visualización de Pandas
pd.options.display.float_format = '{:,.1f}'.format


## 1. Definición Manual de Parámetros
En esta sección definimos los valores base para una unidad (camión) de cada tecnología.


In [2]:
# --- PARÁMETROS DIÉSEL (44t Euro VI) ---
CAPEX_DIESEL = 140_000.0
TARIFA_INTERNA_DIESEL = 1.5   # €/km
OPEX_ANUAL_DIESEL = 158_000.0  # Incluye combustible, mantenimiento, seguros y personal
KM_ANUAL_UNIDAD = 130_000.0

# --- PARÁMETROS ELÉCTRICO (44t BEV) ---
CAPEX_EV_BRUTO = 400_000.0
AYUDA_MOVES = 90_000.0
CAPEX_EV_NETO = CAPEX_EV_BRUTO - AYUDA_MOVES
TARIFA_INTERNA_EV = 1.6       # €/km (Tarifa incentivada por descarbonización)
OPEX_ANUAL_EV = 125_000.0     # Menor coste energético y mantenimiento


## 2. Cálculo del Margen de Contribución Anual (MCA)
El MCA es el beneficio operativo anual que genera cada camión para pagar su propio CAPEX.
$$ MCA = (Tarifa \times Kms) - OPEX $$


In [3]:
# Cálculos Diésel
ingresos_anuales_d = TARIFA_INTERNA_DIESEL * KM_ANUAL_UNIDAD
mca_diesel = ingresos_anuales_d - OPEX_ANUAL_DIESEL

# Cálculos Eléctrico
ingresos_anuales_e = TARIFA_INTERNA_EV * KM_ANUAL_UNIDAD
mca_ev = ingresos_anuales_e - OPEX_ANUAL_EV

print(f"MCA Diésel: {format_1_dec(mca_diesel)} €/año")
print(f"MCA Eléctrico: {format_1_dec(mca_ev)} €/año")


MCA Diésel: 37,000.0 €/año
MCA Eléctrico: 83,000.0 €/año


## 3. Punto de Equilibrio (Break-Even)
Calculamos cuántos años y cuántos viajes se necesitan para recuperar la inversión inicial.


In [4]:
# Viajes promedio al año por camión (asumido)
VIAJES_ANUALES = 250.0

# Break-Even Diésel
be_anos_d = CAPEX_DIESEL / mca_diesel
be_viajes_d = be_anos_d * VIAJES_ANUALES

# Break-Even Eléctrico
be_anos_e = CAPEX_EV_NETO / mca_ev
be_viajes_e = be_anos_e * VIAJES_ANUALES

# Tabla Comparativa
resumen = pd.DataFrame({
    "Concepto": ["CAPEX Neto (€)", "Tarifa (€/km)", "MCA (€/año)", "BE (Años)", "BE (Viajes Total)"],
    "Diésel": [CAPEX_DIESEL, TARIFA_INTERNA_DIESEL, mca_diesel, be_anos_d, be_viajes_d],
    "Eléctrico": [CAPEX_EV_NETO, TARIFA_INTERNA_EV, mca_ev, be_anos_e, be_viajes_e]
}).set_index("Concepto")

display(resumen)


,Diésel,Eléctrico
Concepto,,
CAPEX Neto (€),"140,000.0","310,000.0"
Tarifa (€/km),1.5,1.6
MCA (€/año),"37,000.0","83,000.0"
BE (Años),3.8,3.7
BE (Viajes Total),945.9,933.7


## 4. Análisis de Sensibilidad (Variación de Tarifa)
¿Cómo cambia el tiempo de recuperación si ajustamos la tarifa de transferencia interna?


In [5]:
tarifas_test = [1.3, 1.4, 1.5, 1.6, 1.7, 1.8]
sensibilidad = []

for t in tarifas_test:
    # Diesel
    mca_d = (t * KM_ANUAL_UNIDAD) - OPEX_ANUAL_DIESEL
    be_d = CAPEX_DIESEL / mca_d if mca_d > 0 else np.inf
    
    # EV
    mca_e = (t * KM_ANUAL_UNIDAD) - OPEX_ANUAL_EV
    be_e = CAPEX_EV_NETO / mca_e if mca_e > 0 else np.inf
    
    sensibilidad.append({
        "Tarifa (€/km)": t,
        "BE Diésel (Años)": be_d,
        "BE Eléctrico (Años)": be_e
    })

df_sens = pd.DataFrame(sensibilidad).set_index("Tarifa (€/km)")
display(df_sens)


,BE Diésel (Años),BE Eléctrico (Años)
Tarifa (€/km),,
1.3,12.7,7.0
1.4,5.8,5.4
1.5,3.8,4.4
1.6,2.8,3.7
1.7,2.2,3.2
1.8,1.8,2.8
